# nb_B — Interactive DUNE background inspector

Reads a JSON IR produced by `bki-extract` and renders:
- an interactive 3D Plotly view (z=beam, x=drift, y=vert)
- three 2D projections (Z–Y beam face, Z–X top view, X–Y side view)

**Configure** `IR_FILE` in the cell below, or set the `BKI_IR_FILE` environment variable
before launching the notebook (used by the command-line test script).

In [ ]:
import os
from pathlib import Path

# BKI_IR_FILE env var lets the cmdline test script inject the path
IR_FILE = Path(os.environ.get("BKI_IR_FILE", "../ir.json"))

In [ ]:
from collections import Counter
from duneradgeo.ir import IRCollection

ir = IRCollection.load(IR_FILE)
print(f"Loaded {len(ir.volumes)} volumes")
print(f"  detector : {ir.detector!r}")
print(f"  source   : {Path(ir.source).name}")
print()
for kind, n in sorted(Counter(v.kind for v in ir.volumes).items()):
    print(f"  {kind:<10}  {n} placements")

In [ ]:
from duneradgeo.notebook_plot import make_figure

fig = make_figure(ir)
fig.show()

## Filter by kind

Use `ir.by_kind(...)` to build a sub-view:

In [ ]:
from duneradgeo.ir import IRCollection
from duneradgeo.notebook_plot import make_figure

ir_det = IRCollection(
    source=ir.source,
    detector=ir.detector,
    volumes=ir.by_kind("anode", "opdet", "struct"),
)
make_figure(ir_det).show()

## Static matplotlib PDF

Calls `bki-plot` logic directly and saves a PDF alongside the IR file:

In [ ]:
from duneradgeo.plot import plot_ir

out_pdf = str(IR_FILE.with_suffix(".pdf"))
plot_ir(ir, out_pdf,
        title=f"{Path(ir.source).name} — {ir.detector.upper()} layout")
print(f"Saved to {out_pdf}")